# Visualize fabric batches (KD-tree spatial partition)

Shows the KD-tree recursive bisection that `aggregate.batching.spatial_batch()`
applies to the project fabric. Each batch is a spatially contiguous group of
HRUs that gdptools processes together with its own cached weight CSV at
`<project>/weights/<source>_batch<N>.csv`.

`BATCH_SIZE` below controls the target batch size. The **pipeline default is
500**; this notebook uses **10000** for a legible slide-scale figure (~64
contiguous batches on the gfv2 fabric of ~361k HRUs). The partitioning is
deterministic: same fabric + same `BATCH_SIZE` -> identical HRU membership per
batch (only the integer `batch_id` labels would permute if the fabric's row
order changed, since labels are assigned in DFS order over the recursion).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from nhf_spatial_targets.aggregate.batching import spatial_batch

from _helpers import load_fabric, load_project_paths, save_figure

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# KD-tree target batch size. Pipeline default is 500; 10000 gives a
# slide-legible ~64-batch partition on the gfv2 fabric.
BATCH_SIZE = 10000

# Set True (and re-run) to populate docs/figures/fabric/<project>/*.png
import _helpers

_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

print(f"Project : {project_dir}")
print(f"Fabric  : {fabric_cfg['path']}")
print(f"  HRUs  : {len(fabric):,}")
print(f"  CRS   : {fabric.crs} (reprojected to Albers for plotting)")

## Partition the fabric

`spatial_batch` returns the fabric with an added integer `batch_id` column.
Actual batch sizes vary because the recursion stops when a partition drops
below `batch_size // 2`.

In [ ]:
batched = spatial_batch(fabric, batch_size=BATCH_SIZE)

sizes = batched.groupby("batch_id").size().to_numpy()
n_batches = len(sizes)
print(f"n_batches : {n_batches}")
print(f"min size  : {sizes.min()}")
print(f"max size  : {sizes.max()}")
print(f"mean size : {sizes.mean():.1f}")
print(f"median    : {np.median(sizes):.0f}")

## Main figure: HRU centroids colored by batch

Plotting every polygon is slow; centroids give the same spatial story at
scatter speed. Colors are a random permutation of `batch_id` modulo 20 so
spatially-adjacent batches (which are *not* adjacent in `batch_id`, since the
DFS traversal walks subtrees) end up with contrasting colors. Fixed seed for
reproducibility.

In [ ]:
centroids = batched.geometry.centroid
rng = np.random.default_rng(seed=0)
perm = rng.permutation(n_batches)
color_idx = perm[batched["batch_id"].to_numpy()] % 20

fig, ax = plt.subplots(figsize=(14, 9))
ax.scatter(
    centroids.x,
    centroids.y,
    c=color_idx,
    cmap="tab20",
    s=2.5,
    marker=".",
    linewidths=0,
)
ax.set_aspect("equal")
ax.set_title(
    f"{PROJECT_DIR.name} fabric \u2014 {n_batches} spatial batches "
    f"(KD-tree, batch_size={BATCH_SIZE:,}, {len(batched):,} HRUs, "
    f"CRS {fabric.crs.to_string()})"
)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
plt.tight_layout()
save_figure(fig, "fabric_batches")
plt.show()

## Batch-size distribution

The KD-tree targets `BATCH_SIZE` but actual sizes vary because the recursion
stops when a partition drops below `BATCH_SIZE // 2`. A sanity check that no
batch is pathologically tiny or huge.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sizes, bins=40, edgecolor="black", linewidth=0.5)
ax.axvline(BATCH_SIZE, color="red", linestyle="--", label=f"target ({BATCH_SIZE:,})")
ax.axvline(
    BATCH_SIZE // 2,
    color="orange",
    linestyle=":",
    label=f"min ({BATCH_SIZE // 2:,})",
)
ax.set_xlabel("HRUs per batch")
ax.set_ylabel("count")
ax.set_title(f"{PROJECT_DIR.name} batch size distribution (batch_size={BATCH_SIZE:,})")
ax.legend()
plt.tight_layout()
save_figure(fig, "fabric_batch_sizes")
plt.show()